In [1]:
%matplotlib notebook

In [2]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import json
import jax
import jax.numpy as jnp
import numpy as onp
import matplotlib.pyplot as plt
import time

from msmjax.kernels import split_one_over_r, SoftenerOneOverR
from msmjax.core.shortrange import make_compute_U_zero_with_neighborlist, \
    make_compute_U_and_f_zero_with_neighborlist
from msmjax.bspline.gridops import set_up_grids_all_levels
from msmjax.bspline.gridops import create_compute_U_oneplus_via_potential

import sys

sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmfornn.splines.nesting import compute_J_zeroplus
from msmfornn.gridtools import construct_grids_all_levels
from msmfornn.splines.coefficients import compute_coeffs_withtruncation
from msmfornn.grid_to_grid_mapping import \
    compute_kernel_stencils_all_gridlevels
from msmfornn.helpers.algoparam_choice import suggest_p, suggest_max_gridlevel_nonPBC


# Definitions

## Helper functions

### Exact calculation

In [3]:
def make_compute_U_zero_reference(kernels, periodic: bool, box_lengths=None):
    k_0 = kernels[0]
    sum_of_higher_kernels_at_zero = jnp.sum(
        jnp.asarray([k(0.0) for k in kernels[1:]])
    )
    
    if periodic and box_lengths is None:
        raise ValueError("`box_lengths` are required in periodic case.")
    
    def compute_U_zero_reference(positions, charges):
        R_ij = positions[:, jnp.newaxis, :] - positions
        if periodic:
            R_ij -= jnp.rint(R_ij / box_lengths) * box_lengths
        qi_qj = charges[:, jnp.newaxis] * charges
        indices_triu = jnp.triu_indices(positions.shape[0], k=1)
        r_ij_triu = jnp.linalg.norm(R_ij[indices_triu], axis=1)
        qi_qj_triu = qi_qj[indices_triu]
        
        pair_term = (qi_qj_triu * jax.vmap(k_0)(r_ij_triu)).sum()
        self_energy_term = 0.5 * jnp.diag(qi_qj).sum() * sum_of_higher_kernels_at_zero
    
        return pair_term - self_energy_term
    
    return compute_U_zero_reference

In [4]:
@jax.jit
def calc_total_e_ref_jax(positions, charges):
    R_ij = positions[:, jnp.newaxis, :] - positions
    qi_qj = charges[:, jnp.newaxis] * charges
    indices_triu = jnp.triu_indices(positions.shape[0], k=1)
    r_ij_triu = jnp.linalg.norm(R_ij[indices_triu], axis=1)
    qi_qj_triu = qi_qj[indices_triu]
    
    return (qi_qj_triu * 1. / r_ij_triu).sum()

@jax.jit
def calc_total_f_ref(positions, charges):
    return -jax.grad(calc_total_e_ref_jax, argnums=0)(positions, charges)

@jax.jit
def calc_total_e_and_f_ref(positions, charges):
    value, grad =  jax.value_and_grad(calc_total_e_ref_jax, argnums=0)(positions, charges)
    return value, -grad

In [5]:
def calc_total_e_ref_numpy(positions, charges):
    R_ij = positions[:, onp.newaxis, :] - positions
    qi_qj = charges[:, onp.newaxis] * charges
    indices_triu = onp.triu_indices(positions.shape[0], k=1)
    r_ij_triu = onp.linalg.norm(R_ij[indices_triu], axis=1)
    qi_qj_triu = qi_qj[indices_triu]
    
    return (qi_qj_triu * 1. / r_ij_triu).sum()

### MSM

In [6]:
def suggest_msm_params_nonperiodic(
    box_lengths,
    n_particles,
    level_one_gridspacing,
    level_zero_cutoff,
    p=None,
    mu=None,
    n_levels=None,
):
    alpha = level_zero_cutoff / level_one_gridspacing
    if p is None:
        p = suggest_p(alpha)
    # See section "1. Preprocessing" of the article
    if mu is None:
        mu = max(int(4 * alpha + p // 2), 3 * p // 2)
    if n_levels is None:
        n_levels = suggest_max_gridlevel_nonPBC(
            min_pos=onp.zeros_like(box_lengths),
            max_pos=box_lengths,
            nb_particles=n_particles,
            level_one_gridspacing=level_one_gridspacing,
            level_zero_cutoff=level_zero_cutoff,
            p=p,
        )
        
    return {
        "level_one_gridspacing": level_one_gridspacing,
        "level_zero_cutoff": level_zero_cutoff,
        "p": p,
        "mu": mu,
        "n_levels": n_levels,
    }


def set_up_grids_and_kernels(
    box_lengths,
    pbcs,
    level_one_gridspacing,
    level_zero_cutoff,
    p,
    mu,
    n_levels,
):
    n_dim = len(pbcs)

    kernels = split_one_over_r(
        max_level=n_levels,
        level_zero_cutoff=level_zero_cutoff,
        softening_function=SoftenerOneOverR(p),
    )
    grids = set_up_grids_all_levels(
        box_lengths=box_lengths,
        level_one_spacings=[level_one_gridspacing] * n_dim,
        pbcs=pbcs,
        n_levels=n_levels,
        p=p,
        J_zeroplus=wrapper_old_compute_J_zeroplus(p),
    )
    kernel_stencils = wrapper_old_construct_kernel_stencils(
        kernels=kernels,
        box_lengths=box_lengths,
        level_one_gridspacing=level_one_gridspacing,
        level_zero_cutoff=level_zero_cutoff,
        n_levels=n_levels,
        p=p,
        mu=mu,
    )

    return kernels, grids, kernel_stencils

In [7]:
def make_wrapped_compute_U_zero(kernels, cutoff, box_lengths, pbcs, neighborlist_reference_positions):
    pbcs = jnp.asarray(pbcs)
    if not (jnp.all(pbcs) or jnp.all(~pbcs)):
        raise ValueError("Mixed boundary conditions currently not supported.")
    periodic = pbcs[0]
    
    # Too large box causes the neighbor list functions to throw strange errors.
    # But in the non-periodic case, the box size is actually irrelevant,
    # and we still get the correct result, and no error, by simply
    # specifying a fictitious small box.
    if not periodic:
        box_lengths = jnp.ones_like(box_lengths)
        
    neighbor_fun, compute_U_zero_with_neighborlist = make_compute_U_zero_with_neighborlist(
        kernels=kernels,
        cutoff=cutoff,
        box_lengths=box_lengths,
        pbcs=PBCS,
    )
    neighbor_list = neighbor_fun.allocate(neighborlist_reference_positions)
    
    def wrapped_compute_U_zero(positions, charges):
        updated_neighbor_list = neighbor_fun.update(positions, neighbor_list)
        return compute_U_zero_with_neighborlist(positions, charges, updated_neighbor_list.idx)
    
    return wrapped_compute_U_zero


def make_wrapped_compute_U_and_f_zero(kernels, cutoff, box_lengths, pbcs, neighborlist_reference_positions):
    pbcs = jnp.asarray(pbcs)
    if not (jnp.all(pbcs) or jnp.all(~pbcs)):
        raise ValueError("Mixed boundary conditions currently not supported.")
    periodic = pbcs[0]
    
    # Too large box causes the neighbor list functions to throw strange errors.
    # But in the non-periodic case, the box size is actually irrelevant,
    # and we still get the correct result, and no error, by simply
    # specifying a fictitious small box.
    if not periodic:
        box_lengths = jnp.ones_like(box_lengths)
        
    neighbor_fun, compute = make_compute_U_and_f_zero_with_neighborlist(
        kernels=kernels,
        cutoff=cutoff,
        box_lengths=box_lengths,
        pbcs=PBCS,
    )
    neighbor_list = neighbor_fun.allocate(neighborlist_reference_positions)
    
    def wrapped_compute(positions, charges):
        updated_neighbor_list = neighbor_fun.update(positions, neighbor_list)
        return compute(positions, charges, updated_neighbor_list.idx)
    
    return wrapped_compute

In [8]:
def wrapper_old_compute_J_zeroplus(p):
    return compute_J_zeroplus(p)

def wrapper_old_construct_kernel_stencils(
    kernels,
    box_lengths,
    level_one_gridspacing,
    level_zero_cutoff,
    n_levels,
    p,
    mu,
):
    omega, _ = compute_coeffs_withtruncation(p=p, mu=mu)
    omega_zeroplus = omega[len(omega) // 2 :]
    grids_oldmsm = construct_grids_all_levels(
        min_pos=onp.zeros_like(box_lengths),
        max_pos=box_lengths,
        p=p,
        level_one_gridspacing=level_one_gridspacing,
        max_gridlevel=n_levels,
    )
    kernel_stencils_nonnegative = compute_kernel_stencils_all_gridlevels(
        kernelfunctions=kernels,
        grids=grids_oldmsm,
        level_zero_cutoff=level_zero_cutoff,
        omega_zeroplus=omega_zeroplus,
    )
    kernel_stencils = [None]
    for stncl in kernel_stencils_nonnegative[1:]:
        pw = [(s - 1, 0) for s in stncl.shape]
        stncl_symm = jnp.pad(stncl, pad_width=pw, mode="reflect")
        kernel_stencils.append(stncl_symm)

    return kernel_stencils


In [9]:
def make_compute_U_oneplus(
    kernels,
    level_one_gridspacing,
    level_zero_cutoff,
    n_levels,
    box_lengths,
    pbcs,
    p,
    mu,
    convolution_methods=None,
):
    n_dim = len(pbcs)

    J_zeroplus = wrapper_old_compute_J_zeroplus(p)
    grids = set_up_grids_all_levels(
        box_lengths=box_lengths,
        level_one_spacings=[level_one_gridspacing] * n_dim,
        pbcs=pbcs,
        n_levels=n_levels,
        p=p,
        J_zeroplus=J_zeroplus,
    )
    kernel_stencils = wrapper_old_construct_kernel_stencils(
        kernels=kernels,
        box_lengths=box_lengths,
        level_one_gridspacing=level_one_gridspacing,
        level_zero_cutoff=level_zero_cutoff,
        n_levels=n_levels,
        p=p,
        mu=mu,
    )

    calculate = create_compute_U_oneplus_via_potential(
        grids=grids,
        kernel_stencils=kernel_stencils,
        convolution_methods=convolution_methods,
    )

    return calculate


In [10]:
def set_up_msm(
    level_one_gridspacing,
    level_zero_cutoff,
    box_lengths,
    pbcs,
    neighborlist_reference_positions,
    n_levels=None,
    p=None,
    mu=None,
    conv_meth=None,
    **neighbor_kwargs,
):
    pbcs = jnp.asarray(pbcs)
    if pbcs.any():
        raise ValueError(
            "Periodic or mixed boundary conditions currently not supported."
        )

    alpha = level_zero_cutoff / level_one_gridspacing
    if p is None:
        p = suggest_p(alpha)

    # TODO: mu
    # See section "1. Preprocessing" of the article
    if mu is None:
        mu = max(int(4 * alpha + p // 2), 3 * p // 2)

    # TODO: n_levels
    if n_levels is None:
        n_levels = suggest_max_gridlevel_nonPBC(
            min_pos=onp.zeros_like(box_lengths),
            max_pos=box_lengths,
            nb_particles=neighborlist_reference_positions.shape[0],
            level_one_gridspacing=level_one_gridspacing,
            level_zero_cutoff=level_zero_cutoff,
            p=p,
        )
        
    if conv_meth is None:
        convolution_methods = None
    else:
        convolution_methods = [None] + [conv_meth] * n_levels

    kernels = split_one_over_r(
        max_level=n_levels,
        level_zero_cutoff=level_zero_cutoff,
        softening_function=SoftenerOneOverR(p),
    )
    wrapped_calc_U_zero = make_wrapped_compute_U_zero(
        kernels=kernels,
        cutoff=level_zero_cutoff,
        box_lengths=box_lengths,
        pbcs=pbcs,
        neighborlist_reference_positions=neighborlist_reference_positions,
        **neighbor_kwargs,
    )
    calc_U_oneplus = make_compute_U_oneplus(
        kernels=kernels,
        level_one_gridspacing=level_one_gridspacing,
        level_zero_cutoff=level_zero_cutoff,
        n_levels=n_levels,
        box_lengths=box_lengths,
        pbcs=pbcs,
        p=p,
        mu=mu,
        convolution_methods=convolution_methods,
    )

    def calculate(positions, charges):
        return wrapped_calc_U_zero(positions, charges) + calc_U_oneplus(
            positions, charges
        )
    
    # TODO
    # info = {
    #     "grids": grids, # TODO: return from make_compute_U_oneplus?
    #     "kernel_stencils": kernel_stencils, # TODO: return from make_compute_U_oneplus?
    #     "n_levels": n_levels,
    #     "p": p,
    #     "mu": mu,
    # }

    # return calculate, info    # TODO
    
    return calculate


### Structure generation

In [11]:
class RandomConfigGenerator():
    def __init__(self, avg_interparticle_distance, n_dim, seed=None):
        self.avg_interparticle_distance = avg_interparticle_distance
        self.n_dim = n_dim
        if seed is None:
            self.rng = onp.random.default_rng()
        else:
            self.rng = onp.random.default_rng(seed)
            
    def generate_config(self, n_particles):
        side_length = n_particles ** (1. / self.n_dim) * self.avg_interparticle_distance
        box_lengths = jnp.array([side_length] * self.n_dim)
        pos = self.rng.uniform(low=onp.zeros_like(box_lengths), high=box_lengths, size=(n_particles, self.n_dim))
        chg = self.rng.uniform(low=-1.0, high=1.0, size=n_particles)
        
        return jnp.array(pos), jnp.array(chg), box_lengths
    

## Global settings

In [12]:
# Basic geometry
AVG_NEIGHBOR_DISTANCE = 2.5
N_DIM = 3
PBCS = [False] * N_DIM

# MSM
LEVEL_ONE_GRIDSPACING = AVG_NEIGHBOR_DISTANCE
ALPHA = 3.0
LEVEL_ZERO_CUTOFF = ALPHA * LEVEL_ONE_GRIDSPACING

# Benchmarking
SEED = 55768
N_STRUCTURES_PER_SIZE = 100
NBS_PARTICLES = onp.arange(1, 11) * 1000
# NBS_PARTICLES = onp.arange(1, 4) * 1000

In [13]:
common_msm_args = {
    'level_one_gridspacing': LEVEL_ONE_GRIDSPACING,
    'level_zero_cutoff': LEVEL_ZERO_CUTOFF,
    'pbcs': PBCS,
}

# Benchmark

## Helpers

In [14]:
def set_up_timed_msm_calc_e_incl_nbl(
    box_lengths,
    pbcs,
    neighborlist_ref_pos,
    level_one_gridspacing,
    level_zero_cutoff,
    p,
    mu,
    n_levels,
    conv_meth,
    **neighbor_kwargs,
):
    pbcs = jnp.asarray(pbcs)
    neighborlist_ref_pos = jnp.asarray(neighborlist_ref_pos)

    if conv_meth is None:
        convolution_methods = None
    else:
        convolution_methods = [None] + [conv_meth] * n_levels

    kernels, grids, kernel_stencils = set_up_grids_and_kernels(
        box_lengths=box_lengths,
        pbcs=pbcs,
        level_one_gridspacing=level_one_gridspacing,
        level_zero_cutoff=level_zero_cutoff,
        p=p,
        mu=mu,
        n_levels=n_levels,
    )

    # Too large boxes cause strange errors in neighbor list. But if not periodic,
    # we can still get the correct result, and no error, by passing a fake small box.
    if not pbcs.any():
        box_lengths = jnp.ones_like(box_lengths)
    neighbor_fun, compute_U_zero_nbl = make_compute_U_zero_with_neighborlist(
        kernels=kernels,
        cutoff=level_zero_cutoff,
        box_lengths=box_lengths,
        pbcs=pbcs,
        **neighbor_kwargs,
    )
    neighbor_list = neighbor_fun.allocate(neighborlist_ref_pos)
    compute_U_oneplus = create_compute_U_oneplus_via_potential(
        grids=grids,
        kernel_stencils=kernel_stencils,
        convolution_methods=convolution_methods,
    )

    @jax.jit
    def compute(positions, charges):
        updated_neighbor_list = neighbor_fun.update(positions, neighbor_list)
        U_zero = compute_U_zero_nbl(
            positions, charges, updated_neighbor_list.idx
        )
        U_oneplus = compute_U_oneplus(positions, charges)

        return U_zero + U_oneplus

    def timed_compute(positions, charges):
        t_1 = time.time()
        e = onp.array(compute(positions, charges).block_until_ready())
        t_2 = time.time()
        delta_t = t_2 - t_1
        results = {"energy": e}

        return delta_t, results
    
    info = {"kernels": kernels, "grids": grids, "kernel_stencils": kernel_stencils}

    return timed_compute, info


In [15]:
def set_up_timed_msm_calc_e_excl_nbl(
        box_lengths,
        pbcs,
        neighborlist_ref_pos,
        level_one_gridspacing,
        level_zero_cutoff,
        p,
        mu,
        n_levels,
        conv_meth,
        **neighbor_kwargs,
):
    pbcs = jnp.asarray(pbcs)
    neighborlist_ref_pos = jnp.asarray(neighborlist_ref_pos)

    if conv_meth is None:
        convolution_methods = None
    else:
        convolution_methods = [None] + [conv_meth] * n_levels

    kernels, grids, kernel_stencils = set_up_grids_and_kernels(
        box_lengths=box_lengths,
        pbcs=pbcs,
        level_one_gridspacing=level_one_gridspacing,
        level_zero_cutoff=level_zero_cutoff,
        p=p,
        mu=mu,
        n_levels=n_levels,
    )

    # Too large boxes cause strange errors in neighbor list. But if not periodic,
    # we can still get the correct result, and no error, by passing a fake small box.
    if not pbcs.any():
        box_lengths = jnp.ones_like(box_lengths)
    neighbor_fun, compute_U_zero_nbl = make_compute_U_zero_with_neighborlist(
        kernels=kernels,
        cutoff=level_zero_cutoff,
        box_lengths=box_lengths,
        pbcs=pbcs,
        **neighbor_kwargs,
    )
    neighbor_list = neighbor_fun.allocate(neighborlist_ref_pos)
    nbl_update_fun = jax.jit(neighbor_fun.update)
    compute_U_oneplus = create_compute_U_oneplus_via_potential(
        grids=grids,
        kernel_stencils=kernel_stencils,
        convolution_methods=convolution_methods,
    )

    @jax.jit
    def compute(positions, charges, neighbor_indices):
        U_zero = compute_U_zero_nbl(
            positions, charges, neighbor_indices
        )
        U_oneplus = compute_U_oneplus(positions, charges)

        return U_zero + U_oneplus

    def timed_compute(positions, charges):
        # To exclude the neighbor list update from the timing, we need to call a blocking
        # operation on the neighbor list before.
        # See https://github.com/google/jax/issues/3125 on how to do that for a container structure.
        updated_neighbor_list = nbl_update_fun(positions, neighbor_list)
        _ = jax.tree_util.tree_flatten(updated_neighbor_list)[0][0].block_until_ready()
        t_1 = time.time()
        e = onp.array(compute(positions, charges, updated_neighbor_list.idx).block_until_ready())
        t_2 = time.time()
        delta_t = t_2 - t_1
        results = {"energy": e}

        return delta_t, results

    info = {"kernels": kernels, "grids": grids,
            "kernel_stencils": kernel_stencils}
    
    return timed_compute, info


In [16]:
def make_timed_ref_calc_e_jax(*args, **kwargs):
    def timed_ref_calc_e(positions, charges):
        t_1 = time.time()
        e = onp.array(calc_total_e_ref_jax(positions, charges).block_until_ready())
        t_2 = time.time()
        delta_t = t_2 - t_1
        results = {"energy": e}
    
        return delta_t, results
    
    return timed_ref_calc_e, {}


def make_timed_ref_calc_e_numpy(*args, **kwargs):
    def timed_ref_calc_e(positions, charges):
        positions = onp.asarray(positions)
        charges = onp.asarray(charges)
        t_1 = time.time()
        e = calc_total_e_ref_numpy(positions, charges)
        t_2 = time.time()
        delta_t = t_2 - t_1
        results = {"energy": e}
    
        return delta_t, results
    
    return timed_ref_calc_e, {}
    

In [17]:
 def read_results(resultsfile):
    numbers_of_particles = []
    times = []
    energies = []
    with open(resultsfile, "r") as f:
        for line in f.readlines():
            decoded = json.loads(line)
            numbers_of_particles.append(decoded["n_particles"])
            times.append(onp.array(decoded["times"]))
            energies.append(onp.array(decoded["energies"]))

    return numbers_of_particles, times, energies

## Run timing

In [18]:
CONV_METH = "scipy-fft"

# MAKE_TIMED_COMPUTE = make_timed_ref_calc_e_numpy
# outfile = "timing_results_ref_numpy.txt"
MAKE_TIMED_COMPUTE = make_timed_ref_calc_e_jax
outfile = "timing_results_ref_jax.txt"
# MAKE_TIMED_COMPUTE = set_up_timed_msm_calc_e_incl_nbl
# outfile = "timing_results_msm_incl_nbl.txt"
# MAKE_TIMED_COMPUTE = set_up_timed_msm_calc_e_excl_nbl
# outfile = "timing_results_msm_excl_nbl.txt"

config_gen = RandomConfigGenerator(
    avg_interparticle_distance=AVG_NEIGHBOR_DISTANCE, n_dim=N_DIM, seed=SEED
)

with open(outfile, "w") as of:
    for n_particles in NBS_PARTICLES:
        print("- n_particles =", n_particles)
        pos_initial, chg_initial, box_lengths = config_gen.generate_config(
            n_particles
        )
        msm_params = suggest_msm_params_nonperiodic(
            box_lengths=box_lengths,
            n_particles=n_particles,
            level_one_gridspacing=LEVEL_ONE_GRIDSPACING,
            level_zero_cutoff=LEVEL_ZERO_CUTOFF,
        )
        timed_compute, info = MAKE_TIMED_COMPUTE(
            box_lengths=box_lengths,
            pbcs=PBCS,
            neighborlist_ref_pos=pos_initial,
            **msm_params,
            conv_meth=CONV_METH,
        )
        # call once to trigger jit
        timed_compute(pos_initial, chg_initial)
    
        positions = []
        charges = []
        times = []
        energies = []
        for iteration_number in range(N_STRUCTURES_PER_SIZE):
            pos, chg, _ = config_gen.generate_config(n_particles)
            jax.device_put(pos)
            jax.device_put(chg)
            delta_t, out = timed_compute(pos, chg)
            # TODO: check for neighbor list buffer overflows?
            positions.append(pos)
            charges.append(chg)
            times.append(delta_t)
            energies.append(out["energy"])
    
        output = {
            "n_particles": int(n_particles),
            # "box_lengths": onp.asarray(box_lengths).tolist(),
            # "positions": onp.asarray(positions).tolist(),
            # "charges": onp.asarray(charges).tolist(),
            "times": onp.asarray(times).tolist(),
            "energies": onp.asarray(energies).tolist(),
            "msm_params": msm_params,
        }
        json.dump(output, of)
        of.write("\n")


- n_particles = 1000
Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Highest-level grid not coarser than simulation box size ("criterion 1.")
*) Cutoff at highest-but-one level not too large relative to simulation box size ("criterion 2.")

- n_particles = 2000
Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Highest-level grid not coarser than simulation box size ("criterion 1.")
*) Cutoff at highest-but-one level not too large relative to simulation box size ("criterion 2.")

- n_particles = 3000
Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Highest-level grid not coarser than simulation box size ("criterion 1.")
*) Cutoff at highest-but-one level not too large relati

In [19]:
resultsfiles = [
    # "timing_results_ref_numpy.txt",
    "timing_results_ref_jax.txt",
    "timing_results_msm_incl_nbl.txt",
    "timing_results_msm_excl_nbl.txt",
]
labels = [
    # "Exact calculation, numpy",
    "Exact calculation, jax",
    "MSM, timing incl. neighbor update",
    "MSM, timing excl. neighbor update",
]

fig, ax = plt.subplots()
ax.set_xlabel("Number of particles")
ax.set_ylabel("Time / ms")
for rfile, lbl in zip(resultsfiles, labels):
    nbs_particles, times_all, energies_all = read_results(rfile)
    ax.errorbar(
        nbs_particles,
        onp.mean(times_all, axis=1) * 1000,
        yerr=onp.std(times_all, axis=1) * 1000,
        label=lbl,
        fmt="o",
    )    

ax.legend()
plt.show()

fig.savefig("timings_clean.pdf")


<IPython.core.display.Javascript object>

In [20]:
configs = onp.load("structures/n_particles_1000.npz")

In [21]:
[k for k in configs.keys()]

['n_particles', 'box_lengths', 'positions', 'charges']

In [22]:
configs["charges"].shape

(11, 1000)